# QAOA-GPT inference demo

In [1]:
from model_interface import QAOA_GPT

In [2]:
import pandas as pd
import networkx as nx
import random
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## Loading a model

In [63]:
qaoa_gpt = QAOA_GPT(
    model_ckpt='nanoGPT/out-10_nodes_feather/llama_ckpt_4000_feather_ar_0_85744__er_0_0.pt',
    data_dir='nanoGPT/data/10_nodes_feather', # to take meta.pkl file 
    temp_folder='temp_data',
)

In [64]:
# qaoa_gpt = QAOA_GPT(
#     model_ckpt='nanoGPT/models/n10w_qaoa_mixer/ckpt_16000_gemb__ar_0_96584__er_0_0.pt',
#     data_dir='nanoGPT/models/n10w_qaoa_mixer/data', # to take meta.pkl file 
#     temp_folder='temp_data',
# )

## Generating random graphs

In [65]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [66]:
tqdm.pandas()

Modify this nodes here to match the model before run

In [67]:
n_graphs = 5
n_nodes = 10

In [68]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [69]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 2, {'weight': 0.56}), (0, 3, {'weight': 0.08}), (0, 4, {'weight': 0.38}), (0, 5, {'weight': 0.91}), (0, 6, {'weight': 0.29}), (0, 9, {'weight': 0.34}), (1, 2, {'weight': 0.68}), (1, 4, {'weight': 0.44}), (1, 5, {'weight': 0.72}), (1, 6, {'weight': 0.52}), (1, 7, {'weight': 0.28}), (1, 9, {'weight': 0.92}), (2, 4, {'weight': 0.71}), (2, 5, {'weight': 0.24}), (2, 6, {'weight': 0.71}), (2, 7, {'weight': 0.6}), (2, 8, {'weight': 0.36}), (2, 9, {'weight': 0.9}), (3, 6, {'weight': 0.83}), (3, 7, {'weight': 0.47}), (3, 9, {'weight': 0.37}), (4, 5, {'weight': 0.81}), (4, 8, {'weight': 0.81}), (4, 9, {'weight': 0.76}), (5, 6, {'weight': 0.65}), (5, 7, {'weight': 0.99}), (5, 8, {'weight': 0.71}), (5, 9, {'weight': 0.53}), (6, 7, {'weight': 0.26}), (6, 8, {'weight': 0.73}), (6, 9, {'weight': 0.8}), (7, 8, {'weight': 0.07})])

## Generate circuits with QAOA-GPT model

In [70]:
embedding_method = qaoa_gpt.embedding_method
print(f"Using embedding method: {embedding_method}")

qaoa_gpt_circ_df = qaoa_gpt.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    # calculate_classic_maxcut=True, # to create col enery_gurobi. Default:True
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
)

Using embedding method: feather


Preparing graphs...: 100%|██████████| 5/5 [00:00<00:00, 180.71it/s]


Performing feather embedding


100%|██████████| 5/5 [00:00<00:00, 1009.27it/s]
Inference. Current batch: n_edges: 29, n_graphs: 1: 100%|██████████| 5/5 [00:04<00:00,  1.04it/s]


In [71]:
sample_gr = graphs_edgelist_list_dict['er_graph_0'].edges(data=True)
sample_gr

EdgeDataView([(0, 2, {'weight': 0.58}), (0, 3, {'weight': 0.53}), (0, 4, {'weight': 0.96}), (0, 7, {'weight': 0.42}), (0, 8, {'weight': 0.84}), (1, 3, {'weight': 0.59}), (1, 5, {'weight': 0.76}), (1, 6, {'weight': 0.7}), (1, 7, {'weight': 0.66}), (1, 8, {'weight': 0.28}), (1, 9, {'weight': 0.46}), (2, 3, {'weight': 0.64}), (2, 5, {'weight': 0.66}), (2, 6, {'weight': 0.27}), (2, 7, {'weight': 0.77}), (3, 5, {'weight': 0.77}), (3, 6, {'weight': 0.33}), (3, 7, {'weight': 0.31}), (3, 8, {'weight': 0.49}), (3, 9, {'weight': 0.62}), (4, 5, {'weight': 0.21}), (4, 6, {'weight': 0.59}), (4, 7, {'weight': 0.42}), (4, 8, {'weight': 0.56}), (5, 6, {'weight': 0.02}), (5, 7, {'weight': 0.48}), (5, 8, {'weight': 0.84}), (5, 9, {'weight': 0.54}), (6, 9, {'weight': 0.5}), (7, 9, {'weight': 0.74}), (8, 9, {'weight': 0.31})])

In [72]:
len(graphs_edgelist_list_dict['er_graph_0'].edges(data=True))

31

The graph after prediction is shifted by 1 unit. For example, in NetworkX the edge is (0, 2, 0.36), but in this DataFrame it becomes (1, 3, 0.36)

In [73]:
qaoa_gpt_circ_df[:1]

,graph,n_edges,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_gurobi,label,graph_w_jl,graph_weight_norm
0,"[(1, 3), 0.58, (1, 4), 0.53, (1, 5), 0.96, (1, 8), 0.42, (1, 9), 0.84, (2, 4), 0.59, (2, 6), 0.76, (2, 7), 0.7, (2, 8), 0.66, (2, 9), 0.28, (2, 10), 0.46, (3, 4), 0.64, (3, 6), 0.66, (3, 7), 0.27, (3, 8), 0.77, (4, 6), 0.77, (4, 7), 0.33, (4, 8), 0.31, (4, 9), 0.49, (4, 10), 0.62, (5, 6), 0.21, (5, 7), 0.59, (5, 8), 0.42, (5, 9), 0.56, (6, 7), 0.02, (6, 8), 0.48, (6, 9), 0.84, (6, 10), 0.54, (7, 10), 0.5, (8, 10), 0.74, (9, 10), 0.31]",31,"[[new_layer_p, 11.0, -0.52, 0.29, new_layer_p, 11.0, -0.4, 0.66, new_layer_p, 11.0, -0.34, 0.73, new_layer_p, 11.0, -0.3, 0.67, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.26, 0.88, new_layer_p, 11.0, -0.24, 0.88, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.18, 0.93, new_layer_p, 11.0, -0.12, 1.06], [new_layer_p, 11.0, -0.52, 0.3, new_layer_p, 11.0, -0.4, 0.66, new_layer_p, 11.0, -0.34, 0.77, new_layer_p, 11.0, -0.29, 0.74, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.26, 0.74, new_layer_p, 11.0, -0.25, 0.88, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.12, 1.15, new_layer_p, 11.0, -0.12, 1.15], [new_layer_p, 11.0, -0.52, 0.29, new_layer_p, 11.0, -0.4, 0.66, new_layer_p, 11.0, -0.34, 0.73, new_layer_p, 11.0, -0.3, 0.67, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.26, 0.85, new_layer_p, 11.0, -0.25, 0.88, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.18, 0.93, new_layer_p, 11.0, -0.12, 1.15, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.18, 1.06], [new_layer_p, 11.0, -0.52, 0.29, new_layer_p, 11.0, -0.4, 0.66, new_layer_p, 11.0, -0.34, 0.73, new_layer_p, 11.0, -0.33, 0.77, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.27, 0.88, new_layer_p, 11.0, -0.25, 0.88, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.12, 1.15, new_layer_p, 11.0, -0.12, 1.15, new_layer_p, 11.0, -0.14, 0.88], [new_layer_p, 11.0, -0.52, 0.31, new_layer_p, 11.0, -0.38, 0.77, new_layer_p, 11.0, -0.34, 0.77, new_layer_p, 11.0, -0.31, 0.77, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.26, 0.77, new_layer_p, 11.0, -0.25, 0.88, new_layer_p, 11.0, -0.18, 0.88, new_layer_p, 11.0, -0.18, 0.93, new_layer_p, 11.0, -0.12, 0.94, new_layer_p, 11.0, -0.12, 1.15]]",[],None,er_graph_0,-12.38,test_interactive,"[[1, 3, 0.58], [1, 4, 0.53], [1, 5, 0.96], [1, 8, 0.42], [1, 9, 0.84], [2, 4, 0.59], [2, 6, 0.76], [2, 7, 0.7], [2, 8, 0.66], [2, 9, 0.28], [2, 10, 0.46], [3, 4, 0.64], [3, 6, 0.66], [3, 7, 0.27], [3, 8, 0.77], [4, 6, 0.77], [4, 7, 0.33], [4, 8, 0.31], [4, 9, 0.49], [4, 10, 0.62], [5, 6, 0.21], [5, 7, 0.59], [5, 8, 0.42], [5, 9, 0.56], [6, 7, 0.02], [6, 8, 0.48], [6, 9, 0.84], [6, 10, 0.54], [7, 10, 0.5], [8, 10, 0.74], [9, 10, 0.31]]",1.0


In [74]:
qaoa_gpt_circ_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   graph              5 non-null      object 
 1   n_edges            5 non-null      int64  
 2   q_circuits         5 non-null      object 
 3   adapt_circuit      5 non-null      object 
 4   adapt_full_ar      0 non-null      object 
 5   graph_prefix       5 non-null      object 
 6   energy_gurobi      5 non-null      float64
 7   label              5 non-null      object 
 8   graph_w_jl         5 non-null      object 
 9   graph_weight_norm  5 non-null      float64
dtypes: float64(2), int64(1), object(7)
memory usage: 528.0+ bytes


## Evaluate circuits

In [75]:
qaoa_gpt_circ_eval_df = qaoa_gpt.eval_circ_df_jl(qaoa_gpt_circ_df)


===== DEBUG INFO =====
CWD: /home/mrzaizai2k/code_bao/ADAPT_GPT
Command:
/opt/julia-1.12.1/bin/julia -t 4 --project=/home/mrzaizai2k/code_bao/ADAPT_GPT/ADAPT.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/adapt_gpt_eval_energy.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-03-16__16_20_14_df.json /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-03-16__16_20_14_df_jl.json 10 qaoa_double_pool




Julia return code: 0


In [76]:
sample_gr

EdgeDataView([(0, 2, {'weight': 0.58}), (0, 3, {'weight': 0.53}), (0, 4, {'weight': 0.96}), (0, 7, {'weight': 0.42}), (0, 8, {'weight': 0.84}), (1, 3, {'weight': 0.59}), (1, 5, {'weight': 0.76}), (1, 6, {'weight': 0.7}), (1, 7, {'weight': 0.66}), (1, 8, {'weight': 0.28}), (1, 9, {'weight': 0.46}), (2, 3, {'weight': 0.64}), (2, 5, {'weight': 0.66}), (2, 6, {'weight': 0.27}), (2, 7, {'weight': 0.77}), (3, 5, {'weight': 0.77}), (3, 6, {'weight': 0.33}), (3, 7, {'weight': 0.31}), (3, 8, {'weight': 0.49}), (3, 9, {'weight': 0.62}), (4, 5, {'weight': 0.21}), (4, 6, {'weight': 0.59}), (4, 7, {'weight': 0.42}), (4, 8, {'weight': 0.56}), (5, 6, {'weight': 0.02}), (5, 7, {'weight': 0.48}), (5, 8, {'weight': 0.84}), (5, 9, {'weight': 0.54}), (6, 9, {'weight': 0.5}), (7, 9, {'weight': 0.74}), (8, 9, {'weight': 0.31})])

The eval df add 1 more col is adapt_gpt_energies

Each graph is generate into 5 other graphs, that why we see list of 5 q_circuits and adapt_gpt_energies



In [77]:
qaoa_gpt_circ_eval_df[:1]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 3], 0.58, [1, 4], 0.53, [1, 5], 0.96, [1, 8], 0.42, [1, 9], 0.84, [2, 4], 0.59, [2, 6], 0.76, [2, 7], 0.7000000000000001, [2, 8], 0.66, [2, 9], 0.28, [2, 10], 0.46, [3, 4], 0.64, [3, 6], 0.66, [3, 7], 0.27, [3, 8], 0.77, [4, 6], 0.77, [4, 7], 0.33, [4, 8], 0.31, [4, 9], 0.49, [4, 10], 0.62, [5, 6], 0.21, [5, 7], 0.59, [5, 8], 0.42, [5, 9], 0.56, [6, 7], 0.02, [6, 8], 0.48, [6, 9], 0.84, [6, 10], 0.54, [7, 10], 0.5, [8, 10], 0.74, [9, 10], 0.31]",31,"[[new_layer_p, 11, -0.52, 0.29, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.73, new_layer_p, 11, -0.30000000000000004, 0.67, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.88, new_layer_p, 11, -0.24, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.93, new_layer_p, 11, -0.12, 1.06], [new_layer_p, 11, -0.52, 0.30000000000000004, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.77, new_layer_p, 11, -0.29, 0.74, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.74, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.12, 1.15], [new_layer_p, 11, -0.52, 0.29, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.73, new_layer_p, 11, -0.30000000000000004, 0.67, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.85, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.93, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 1.06], [new_layer_p, 11, -0.52, 0.29, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.73, new_layer_p, 11, -0.33, 0.77, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.27, 0.88, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.14, 0.88], [new_layer_p, 11, -0.52, 0.31, new_layer_p, 11, -0.38, 0.77, new_layer_p, 11, -0.34, 0.77, new_layer_p, 11, -0.31, 0.77, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.77, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.93, new_layer_p, 11, -0.12, 0.9400000000000001, new_layer_p, 11, -0.12, 1.15]]","[-11.993487537330491, -11.992428879278243, -11.748958226636939, -11.762500571446466, -11.956423126991039]",-12.38


In [78]:
qaoa_gpt_circ_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   graph_prefix        5 non-null      object 
 1   graph               5 non-null      object 
 2   n_edges             5 non-null      int64  
 3   q_circuits          5 non-null      object 
 4   adapt_gpt_energies  5 non-null      object 
 5   energy_gurobi       5 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 368.0+ bytes


In [79]:
# 3 extract first rows into 5 rows for 5 q_circuits and adapt_gpt_energies
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits']) 

In [80]:
qaoa_gpt_circ_eval_expl_df[:5]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 3], 0.58, [1, 4], 0.53, [1, 5], 0.96, [1, 8], 0.42, [1, 9], 0.84, [2, 4], 0.59, [2, 6], 0.76, [2, 7], 0.7000000000000001, [2, 8], 0.66, [2, 9], 0.28, [2, 10], 0.46, [3, 4], 0.64, [3, 6], 0.66, [3, 7], 0.27, [3, 8], 0.77, [4, 6], 0.77, [4, 7], 0.33, [4, 8], 0.31, [4, 9], 0.49, [4, 10], 0.62, [5, 6], 0.21, [5, 7], 0.59, [5, 8], 0.42, [5, 9], 0.56, [6, 7], 0.02, [6, 8], 0.48, [6, 9], 0.84, [6, 10], 0.54, [7, 10], 0.5, [8, 10], 0.74, [9, 10], 0.31]",31,"[new_layer_p, 11, -0.52, 0.29, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.73, new_layer_p, 11, -0.30000000000000004, 0.67, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.88, new_layer_p, 11, -0.24, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.93, new_layer_p, 11, -0.12, 1.06]",-11.993488,-12.38
0,er_graph_0,"[[1, 3], 0.58, [1, 4], 0.53, [1, 5], 0.96, [1, 8], 0.42, [1, 9], 0.84, [2, 4], 0.59, [2, 6], 0.76, [2, 7], 0.7000000000000001, [2, 8], 0.66, [2, 9], 0.28, [2, 10], 0.46, [3, 4], 0.64, [3, 6], 0.66, [3, 7], 0.27, [3, 8], 0.77, [4, 6], 0.77, [4, 7], 0.33, [4, 8], 0.31, [4, 9], 0.49, [4, 10], 0.62, [5, 6], 0.21, [5, 7], 0.59, [5, 8], 0.42, [5, 9], 0.56, [6, 7], 0.02, [6, 8], 0.48, [6, 9], 0.84, [6, 10], 0.54, [7, 10], 0.5, [8, 10], 0.74, [9, 10], 0.31]",31,"[new_layer_p, 11, -0.52, 0.30000000000000004, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.77, new_layer_p, 11, -0.29, 0.74, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.74, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.12, 1.15]",-11.992429,-12.38
0,er_graph_0,"[[1, 3], 0.58, [1, 4], 0.53, [1, 5], 0.96, [1, 8], 0.42, [1, 9], 0.84, [2, 4], 0.59, [2, 6], 0.76, [2, 7], 0.7000000000000001, [2, 8], 0.66, [2, 9], 0.28, [2, 10], 0.46, [3, 4], 0.64, [3, 6], 0.66, [3, 7], 0.27, [3, 8], 0.77, [4, 6], 0.77, [4, 7], 0.33, [4, 8], 0.31, [4, 9], 0.49, [4, 10], 0.62, [5, 6], 0.21, [5, 7], 0.59, [5, 8], 0.42, [5, 9], 0.56, [6, 7], 0.02, [6, 8], 0.48, [6, 9], 0.84, [6, 10], 0.54, [7, 10], 0.5, [8, 10], 0.74, [9, 10], 0.31]",31,"[new_layer_p, 11, -0.52, 0.29, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.73, new_layer_p, 11, -0.30000000000000004, 0.67, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.26, 0.85, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 0.93, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.18, 1.06]",-11.748958,-12.38
0,er_graph_0,"[[1, 3], 0.58, [1, 4], 0.53, [1, 5], 0.96, [1, 8], 0.42, [1, 9], 0.84, [2, 4], 0.59, [2, 6], 0.76, [2, 7], 0.7000000000000001, [2, 8], 0.66, [2, 9], 0.28, [2, 10], 0.46, [3, 4], 0.64, [3, 6], 0.66, [3, 7], 0.27, [3, 8], 0.77, [4, 6], 0.77, [4, 7], 0.33, [4, 8], 0.31, [4, 9], 0.49, [4, 10], 0.62, [5, 6], 0.21, [5, 7], 0.59, [5, 8], 0.42, [5, 9], 0.56, [6, 7], 0.02, [6, 8], 0.48, [6, 9], 0.84, [6, 10], 0.54, [7, 10], 0.5, [8, 10], 0.74, [9, 10], 0.31]",31,"[new_layer_p, 11, -0.52, 0.29, new_layer_p, 11, -0.4, 0.66, new_layer_p, 11, -0.34, 0.73, new_layer_p, 11, -0.33, 0.77, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.27, 0.88, new_layer_p, 11, -0.25, 0.88, new_layer_p, 11, -0.18, 0.88, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.12, 1.15, new_layer_p, 11, -0.14, 0.88]",-11.762501,-12.38
0,er_graph_0,"[[1, 3], 0.58, [1, 4], 0.53, [1, 5], 0.96, [1, 8], 0.42, [1, 9], 0.84, [2, 4], 0.59, [2, 6], 0.76, [2, 7], 0.7000000000000001, [2, 8], 0.66, [2, 9], 0.28, [2, 10], 0.46, [3, 4], 0.64, [3, 6], 0.66, [3, 7], 0.27, [3, 8], 0.77, [4, 6], 0.77, [4, 7], 0.33, [4, 8], 0.31, [4, 9], 0.49, [4, 10], 0.62, [5, 6], 0.21, [5, 7], 0.59, [5, 8], 0.42, [5, 9], 0.56, [6, 7], 0.02, [6, 8], 0.48, [6, 9], 0.84, [6, 10], 0.54, [7, 10], 0.5, [8, 10], 0.74, [9, 10], 0.31]",31,"[new_layer_p, 11, -0.52, 0.31, new_layer_p, 11, -0.38, 0.77, new_layer_p, 11, -0.34, 0.77, new_layer_p, 11, -0.31, 0.77, n

This computes how close each QAOA-GPT circuit’s energy is to the optimal solution (AR — approximation ratio).

If the ratio = 1 → perfect solution.

If <1 → circuit energy is above the optimal energy (since MaxCut is a minimization problem in some conventions, sometimes they flip it; conceptually, ratio shows performance).

In [81]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(0.9504902994569657)

In [82]:
# on avg, how many layers are there in the predicted circuits
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(10.56)